# ⚠️ ADVERTENCIA: ESTA VERSIÓN UTILIZA INFORMACIÓN FUTURA (LOOK-AHEAD BIAS) ⚠️

## NO DEBE INTERPRETARSE COMO UN BACKTEST REALISTA NI UTILIZARSE PARA TOMAR DECISIONES DE INVERSIÓN

El algoritmo confirma el rebote utilizando el cierre de una sesión, pero simula la compra en la apertura de esa misma sesión. En tiempo real, ese cierre todavía no se conocía cuando se produjo la apertura. Esta versión se conserva únicamente para reproducir y estudiar el algoritmo original.

# Backtester V2.2 configurable SOLO LONG — algoritmo original RSI + Bollinger

Esta versión utiliza el algoritmo de la sección **“ESTRATEGIA META RSI + BOLLINGUER CON VENTANAS SOLAPADAS (el que mejor funciona)”** del cuaderno `Cuaderno_Pruebas_META_y_otros.ipynb`, adaptado para poder elegir hasta 12 activos, sus capitales, las ventanas temporales y la fecha final del estudio.

La lógica de inversión es exclusivamente **LONG**:

- Sobreventa previa: `RSI < 35`.
- El cierre previo toca o atraviesa la banda inferior de Bollinger.
- La sesión actual confirma un rebote cerrando por encima del cierre previo.
- La compra se registra en la apertura de la sesión de confirmación.
- La venta se realiza por stop loss del 5 %, `RSI >= 65`, banda superior de Bollinger, trailing stop del 6 % o final de la ventana.

## Uso

1. En **CONFIGURACIÓN**, activa los valores deseados y asigna su capital.
2. Selecciona las ventanas en `VENTANAS_MESES`.
3. Usa `FECHA_FIN_VENTANA = None` o `'HOY'` para la fecha actual, o escribe una fecha como `'2026-01-01'`.
4. Ejecuta todas las celdas.

Si la fecha final no es una sesión bursátil, se utiliza la última sesión común anterior. Los precios se descargan ajustados por dividendos de forma predeterminada. El modelo no incluye comisiones, deslizamiento ni impuestos.

In [ ]:
# ========================= CONFIGURACIÓN GENERAL =========================
# Esta sección reúne los parámetros que puede modificar el usuario.

# Cada elemento contiene el ticker de Yahoo Finance y su capital asignado.
# Para activar o desactivar un valor, quite o añada # al inicio de su línea.
ACTIVOS = [
    ('META',    5_000),
    ('SHOP',    5_000),
    # ('ITX.MC',  5_000),      # España
    # ('7203.T',  5_000),      # Toyota, Japón
    # ('7974.T',  5_000),      # Nintendo, Japón
    # ('BMW.DE',  5_000),      # Alemania
    # ('MC.PA',   5_000),      # LVMH, Francia
    # ('ASML.AS', 5_000),      # Países Bajos
    # ('AAPL',    5_000),
    # ('NVDA',    5_000),
    # ('MSFT',    5_000),
    # ('AMZN',    5_000),
]

# Duraciones que se analizarán simultáneamente, expresadas en meses.
VENTANAS_MESES = [
    6,
    9
]
# Ejemplo alternativo:
# VENTANAS_MESES = [3, 6, 9, 12, 15, 18]

# Fecha desde la que se cuentan hacia atrás todas las ventanas.
# Opciones: None, 'HOY' o una fecha con formato 'AAAA-MM-DD'.
FECHA_FIN_VENTANA = None
# FECHA_FIN_VENTANA = '2026-01-01'

# Parámetros originales del algoritmo de la celda seleccionada.
AJUSTAR_DIVIDENDOS = True
PERIODO_RSI = 14
PERIODO_BOLLINGER = 20
DESVIACIONES_BOLLINGER = 2
RSI_ENTRADA = 35
RSI_SALIDA = 65
STOP_LOSS_LONG = 0.05
TRAILING_STOP_LONG = 0.06
# ========================================================================

In [ ]:
# ==================== IMPORTACIÓN Y PREPARACIÓN DE DATOS ====================
# Librerías estándar: avisos, carpeta temporal y manejo de rutas.
import warnings
import tempfile
from pathlib import Path

# relativedelta resta meses naturales (no un número aproximado de días).
from dateutil.relativedelta import relativedelta

# Librerías de cálculo, tablas, gráficos y descarga de cotizaciones.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import yfinance as yf
from IPython import get_ipython
from IPython.display import display

# Se ocultan avisos no críticos para que la salida del cuaderno sea legible.
warnings.filterwarnings('ignore')

# yfinance guarda su caché de zonas horarias en la carpeta temporal del sistema.
yf.set_tz_cache_location(str(Path(tempfile.gettempdir()) / 'yf_backtester_v22_config_cache'))

# Activa los gráficos integrados solo cuando el código se ejecuta en Jupyter.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('matplotlib', 'inline')

# ---------------------- Validación de la configuración ----------------------
# Estas comprobaciones detienen pronto el programa y muestran un mensaje claro
# si el usuario ha escrito una configuración imposible o ambigua.
if not 1 <= len(ACTIVOS) <= 12:
    raise ValueError('Debes activar entre 1 y 12 valores.')

# Se normalizan los tickers a mayúsculas y se crea un diccionario ticker-capital.
tickers = [str(t).strip().upper() for t, _ in ACTIVOS]
capitales = {str(t).strip().upper(): float(c) for t, c in ACTIVOS}
if len(set(tickers)) != len(tickers):
    raise ValueError('No puede haber tickers duplicados en ACTIVOS.')
if any(c <= 0 for c in capitales.values()):
    raise ValueError('Todos los capitales deben ser mayores que cero.')
if not VENTANAS_MESES or any(not isinstance(m, int) or m <= 0 for m in VENTANAS_MESES):
    raise ValueError('VENTANAS_MESES debe contener enteros positivos.')

# Se eliminan ventanas repetidas y se ordenan de menor a mayor.
VENTANAS_MESES = sorted(set(VENTANAS_MESES))

# ---------------------- Resolución de la fecha final V2.2 -------------------
# None y el texto HOY significan lo mismo: la fecha actual, sin hora.
if FECHA_FIN_VENTANA is None or str(FECHA_FIN_VENTANA).strip().upper() == 'HOY':
    fecha_fin_solicitada = pd.Timestamp.today().normalize()
else:
    try:
        fecha_fin_solicitada = pd.Timestamp(FECHA_FIN_VENTANA).normalize()
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "FECHA_FIN_VENTANA debe ser None, 'HOY' o una fecha válida 'AAAA-MM-DD'."
        ) from exc

# No se admiten fechas futuras porque todavía no existen cotizaciones para ellas.
hoy = pd.Timestamp.today().normalize()
if fecha_fin_solicitada > hoy:
    raise ValueError('FECHA_FIN_VENTANA no puede ser posterior a la fecha actual.')

# Se descargan cinco meses extra antes de la ventana mayor. Ese margen permite
# calcular RSI y Bollinger sin perder el inicio efectivo del backtest.
max_meses = max(VENTANAS_MESES)
inicio_descarga = fecha_fin_solicitada - relativedelta(months=max_meses + 5)

datos_raw = {}
errores = []

# BUCLE DE DESCARGA: recorre una vez cada ticker activo, descarga sus velas
# diarias, homogeneiza columnas/fechas y guarda el DataFrame ya limpio.
for ticker in tickers:
    try:
        # yfinance interpreta end como límite exclusivo; se suma un día para
        # incluir FECHA_FIN_VENTANA cuando esta sea una sesión bursátil.
        d = yf.download(
            ticker,
            start=inicio_descarga,
            end=fecha_fin_solicitada + pd.Timedelta(days=1),
            interval='1d',
            auto_adjust=AJUSTAR_DIVIDENDOS,
            progress=False,
            threads=False
        )

        # Algunas versiones de yfinance devuelven columnas multinivel incluso
        # al pedir un único ticker. Aquí se convierten en columnas sencillas.
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = d.columns.get_level_values(0)

        # Solo se conservan los campos OHLCV que utiliza la estrategia.
        d = d[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()

        # Se eliminan zona horaria y hora para comparar únicamente fechas.
        if getattr(d.index, 'tz', None) is not None:
            d.index = d.index.tz_localize(None)
        d.index = pd.to_datetime(d.index).normalize()
        d = d[~d.index.duplicated(keep='last')].sort_index()

        # Sesenta sesiones son el mínimo prudente para indicadores y backtest.
        if len(d) < 60:
            raise ValueError(f'solo se recibieron {len(d)} sesiones')
        datos_raw[ticker] = d
    except Exception as exc:
        # No se interrumpe el bucle: se recopilan todos los fallos para informar
        # juntos al usuario al terminar de intentar las descargas.
        errores.append(f'{ticker}: {exc}')

if errores:
    raise RuntimeError('Error descargando datos:\n' + '\n'.join(errores))

# Cada mercado puede tener festivos distintos. El mínimo de sus últimas fechas
# garantiza que todos los activos terminan en una sesión realmente compartida.
ultima_fecha_comun = min(d.index.max() for d in datos_raw.values())

print('Valores activos:', ', '.join(f'{t} ({capitales[t]:,.2f})' for t in tickers))
print('Ventanas activas:', ', '.join(f'{m} meses' for m in VENTANAS_MESES))
print('Fecha final solicitada:', fecha_fin_solicitada.date())
print('Última sesión común utilizada:', ultima_fecha_comun.date())

In [ ]:
# ========================= CÁLCULO DE INDICADORES =========================
def preparar_senales(df):
    """Calcula los indicadores del algoritmo original RSI + Bollinger."""
    d = df.copy()

    # Bollinger: media móvil de 20 sesiones y bandas situadas a dos
    # desviaciones típicas. Se calculan tanto la banda inferior como la superior.
    d['MA20'] = d['Close'].rolling(PERIODO_BOLLINGER).mean()
    d['STD20'] = d['Close'].rolling(PERIODO_BOLLINGER).std()
    d['BB_SUP'] = d['MA20'] + DESVIACIONES_BOLLINGER * d['STD20']
    d['BB_INF'] = d['MA20'] - DESVIACIONES_BOLLINGER * d['STD20']

    # RSI original: medias aritméticas móviles de ganancias y pérdidas de las
    # últimas 14 sesiones. No se sustituye por la variante exponencial de Wilder.
    delta = d['Close'].diff()
    ganancias = delta.clip(lower=0)
    perdidas = -delta.clip(upper=0)
    media_ganancias = ganancias.rolling(PERIODO_RSI).mean()
    media_perdidas = perdidas.rolling(PERIODO_RSI).mean()
    rs = media_ganancias / media_perdidas
    d['RSI'] = 100 - (100 / (1 + rs))

    return d


# BUCLE IMPLÍCITO: prepara de forma independiente los indicadores de cada activo
# y limita todos los históricos a la última fecha común disponible.
datos = {
    ticker: preparar_senales(d.loc[d.index <= ultima_fecha_comun])
    for ticker, d in datos_raw.items()
}

In [ ]:
# ============= MOTOR ORIGINAL RSI + BOLLINGER, ADAPTADO A MULTIACTIVO =============
def ejecutar_v22_solo_long(ticker, fecha_inicio, capital_inicial):
    """Ejecuta para un activo el algoritmo original de la celda 39.

    Se conservan sus reglas: confirmación de rebote mediante el cierre de la
    sesión actual, compra registrada en su apertura y salidas valoradas al cierre.
    """
    d = datos[ticker]
    fechas = d.index[(d.index >= fecha_inicio) & (d.index <= ultima_fecha_comun)]
    df = d.loc[fechas].copy()

    if len(df) < 2:
        raise ValueError(f'{ticker}: datos insuficientes para la ventana solicitada.')

    # Estado inicial de una simulación independiente.
    capital = float(capital_inicial)
    acciones = 0.0
    precio_entrada = None
    fecha_entrada = None
    max_precio = None
    operaciones = []
    equity = []

    # BUCLE PRINCIPAL: comienza en la segunda fila porque la entrada necesita
    # comparar los datos de la sesión actual con los de la sesión anterior.
    for i in range(1, len(df)):
        fecha = df.index[i]
        precio = float(df['Close'].iloc[i])
        apertura = float(df['Open'].iloc[i])
        rsi = float(df['RSI'].iloc[i])
        bb_sup = float(df['BB_SUP'].iloc[i])

        # -------------------------------------------------------------
        # FUERA DEL MERCADO: buscar una nueva entrada LONG.
        # -------------------------------------------------------------
        if acciones == 0:
            rsi_anterior = float(df['RSI'].iloc[i - 1])
            cierre_anterior = float(df['Close'].iloc[i - 1])
            bb_inf_anterior = float(df['BB_INF'].iloc[i - 1])

            # Las dos primeras condiciones detectan sobreventa en la sesión
            # anterior. La tercera exige que el cierre actual confirme el rebote.
            senal = (
                rsi_anterior < RSI_ENTRADA
                and cierre_anterior <= bb_inf_anterior
                and precio > cierre_anterior
            )

            if senal:
                # Se invierte todo el capital disponible al precio de apertura.
                precio_entrada = apertura
                acciones = capital / precio_entrada
                max_precio = precio_entrada
                fecha_entrada = fecha

        # -------------------------------------------------------------
        # DENTRO DEL MERCADO: actualizar máximo y comprobar las salidas.
        # -------------------------------------------------------------
        else:
            max_precio = max(max_precio, precio)
            rentabilidad = (precio / precio_entrada) - 1
            retroceso_desde_maximo = (precio / max_precio) - 1
            motivo = None

            # El orden es el mismo que en el algoritmo original. Si coinciden
            # varias condiciones, se registra únicamente la primera de la lista.
            if rentabilidad <= -STOP_LOSS_LONG:
                motivo = 'STOP LOSS'
            elif rsi >= RSI_SALIDA:
                motivo = f'RSI >= {RSI_SALIDA}'
            elif precio >= bb_sup:
                motivo = 'BOLLINGER SUPERIOR'
            elif retroceso_desde_maximo <= -TRAILING_STOP_LONG:
                motivo = 'TRAILING STOP'

            if motivo is not None:
                # Igual que en la fuente original, la salida se valora al cierre.
                precio_salida = precio
                capital = acciones * precio_salida
                rentabilidad_operacion = (
                    precio_salida / precio_entrada - 1
                ) * 100
                operaciones.append({
                    'Ticker': ticker,
                    'Tipo': 'LONG',
                    'Sistema entrada': 'RSI + BOLLINGER + CONFIRMACIÓN',
                    'Entrada': fecha_entrada,
                    'Precio entrada': precio_entrada,
                    'Salida': fecha,
                    'Precio salida': precio_salida,
                    'Rentabilidad %': rentabilidad_operacion,
                    'Motivo': motivo
                })
                acciones = 0.0
                precio_entrada = None
                fecha_entrada = None
                max_precio = None

        # EQUITY DIARIA: con acciones se valora la posición al cierre; fuera del
        # mercado se conserva el último capital realizado en efectivo.
        valor = acciones * precio if acciones > 0 else capital
        equity.append({'Fecha': fecha, 'Equity': valor})

    # Si queda una compra abierta, se cierra en el último cierre de la ventana.
    if acciones > 0:
        precio_salida = float(df['Close'].iloc[-1])
        capital = acciones * precio_salida
        rentabilidad_operacion = (
            precio_salida / precio_entrada - 1
        ) * 100
        operaciones.append({
            'Ticker': ticker,
            'Tipo': 'LONG',
            'Sistema entrada': 'RSI + BOLLINGER + CONFIRMACIÓN',
            'Entrada': fecha_entrada,
            'Precio entrada': precio_entrada,
            'Salida': df.index[-1],
            'Precio salida': precio_salida,
            'Rentabilidad %': rentabilidad_operacion,
            'Motivo': 'FIN DEL PERIODO'
        })
        equity[-1]['Equity'] = capital

    # Se calcula el drawdown sobre la curva diaria con sus fechas reales.
    eq = pd.DataFrame(equity).set_index('Fecha')
    dd = (eq['Equity'] / eq['Equity'].cummax() - 1).min() * 100

    return {
        'Ticker': ticker,
        'Capital inicial': capital_inicial,
        'Capital final': capital,
        'Rentabilidad %': (capital / capital_inicial - 1) * 100,
        'Drawdown máximo %': dd,
        'Operaciones': operaciones,
        'Equity': eq,
        'Fecha inicio': df.index[0],
        'Fecha fin': df.index[-1]
    }

In [ ]:
# ===== EJECUCIÓN MULTIACTIVO DEL ALGORITMO ORIGINAL RSI + BOLLINGER =====
resultados = {}  # Estructura anidada: resultados[meses][ticker].
filas = []       # Registros que después formarán la tabla comparativa.

# BUCLE EXTERIOR: crea una simulación independiente por cada duración elegida.
# La fecha inicial se calcula hacia atrás desde la fecha final común de V2.2.
for meses in VENTANAS_MESES:
    fecha_objetivo = ultima_fecha_comun - relativedelta(months=meses)
    resultados[meses] = {}

    # BUCLE INTERIOR: dentro de la ventana actual ejecuta todos los tickers con
    # su propio capital. Los activos no comparten dinero ni posiciones.
    for ticker in tickers:
        r = ejecutar_v22_solo_long(ticker, fecha_objetivo, capitales[ticker])
        resultados[meses][ticker] = r
        ops = pd.DataFrame(r['Operaciones'])

        # Cada diccionario será una fila de resultados para activo y ventana.
        filas.append({
            'Ventana (meses)': meses,
            'Ticker': ticker,
            'Capital inicial': r['Capital inicial'],
            'Capital final': r['Capital final'],
            'Beneficio': r['Capital final'] - r['Capital inicial'],
            'Rentabilidad %': r['Rentabilidad %'],
            'Drawdown máximo %': r['Drawdown máximo %'],
            'Operaciones': len(ops),
            'Acierto %': ops['Rentabilidad %'].gt(0).mean() * 100 if not ops.empty else 0
        })

    # Las curvas individuales se alinean por fecha y se suman para construir la
    # cartera. ffill conserva el último valor conocido cuando falta una sesión.
    curvas = [
        r['Equity']['Equity'].rename(ticker)
        for ticker, r in resultados[meses].items()
    ]
    total_eq = pd.concat(curvas, axis=1).sort_index().ffill().dropna().sum(axis=1)
    capital_total = sum(capitales.values())
    final_total = float(total_eq.iloc[-1])
    dd_total = (total_eq / total_eq.cummax() - 1).min() * 100

    # Se añade una fila resumen ponderada por el capital real de cada activo.
    filas.append({
        'Ventana (meses)': meses,
        'Ticker': 'TOTAL CARTERA',
        'Capital inicial': capital_total,
        'Capital final': final_total,
        'Beneficio': final_total - capital_total,
        'Rentabilidad %': (final_total / capital_total - 1) * 100,
        'Drawdown máximo %': dd_total,
        'Operaciones': sum(len(r['Operaciones']) for r in resultados[meses].values()),
        'Acierto %': np.nan
    })
    resultados[meses]['TOTAL CARTERA'] = {'Equity': total_eq.to_frame('Equity')}

# Conversión de registros en una tabla con índice doble: ventana y ticker.
tabla = pd.DataFrame(filas).set_index(['Ventana (meses)', 'Ticker'])
# El título muestra la fecha final elegida en FECHA_FIN_VENTANA. Cuando esa
# fecha no es bursátil, se informa también de la sesión común realmente usada.
titulo_tabla = (
    f'TABLA COMPARATIVA — FECHA FIN DE VENTANA: '
    f'{fecha_fin_solicitada.strftime("%d/%m/%Y")}'
)
if ultima_fecha_comun != fecha_fin_solicitada:
    titulo_tabla += (
        f' — ÚLTIMA SESIÓN UTILIZADA: '
        f'{ultima_fecha_comun.strftime("%d/%m/%Y")}'
    )
print(titulo_tabla)

# El formateo afecta solo a la presentación. La fila total se resalta para que
# pueda distinguirse rápidamente de los resultados de valores individuales.
display(tabla.style.format({
    'Capital inicial': '{:,.2f}',
    'Capital final': '{:,.2f}',
    'Beneficio': '{:+,.2f}',
    'Rentabilidad %': '{:+.2f}%',
    'Drawdown máximo %': '{:+.2f}%',
    'Acierto %': lambda x: '' if pd.isna(x) else f'{x:.1f}%'
}).apply(
    lambda fila: ['font-weight:bold;background-color:#fff3cd'] * len(fila)
    if fila.name[1] == 'TOTAL CARTERA' else [''] * len(fila),
    axis=1
))

In [ ]:
# ================= GRÁFICOS DE PRECIOS Y OPERACIONES LONG =================
# A cada ticker se le asigna un color estable en todos los gráficos.
paleta = plt.get_cmap('tab10')
colores = {ticker: paleta(i % 10) for i, ticker in enumerate(tickers)}

# Verde identifica una compra LONG y naranja identifica su venta de cierre.
COLOR_LONG = '#00A651'
COLOR_SALIDA = '#FF8C00'

# BUCLE DE VENTANAS: produce y guarda una figura independiente por duración.
for meses in VENTANAS_MESES:
    fig, ax = plt.subplots(figsize=(19, 9.5))

    # BUCLE DE ACTIVOS: dibuja cada precio normalizado y sus operaciones. La
    # normalización a 100 permite comparar activos con precios distintos.
    for ticker in tickers:
        r = resultados[meses][ticker]
        d = datos[ticker].loc[r['Fecha inicio']:r['Fecha fin']]
        normalizado = d['Close'] / d['Close'].iloc[0] * 100
        ax.plot(
            normalizado.index, normalizado, color=colores[ticker],
            lw=2.8, alpha=.92, label=ticker, zorder=2
        )

        # BUCLE DE OPERACIONES: marca con un triángulo cada compra y con una X
        # cada venta. Todas las entradas son LONG en esta variante.
        for op in r['Operaciones']:
            if op['Entrada'] in normalizado.index:
                y_entrada = float(normalizado.loc[op['Entrada']])
                ax.scatter(
                    op['Entrada'], y_entrada, color=COLOR_LONG,
                    marker='^', s=190, edgecolor='black', linewidth=1.2,
                    zorder=7
                )

            if op['Salida'] in normalizado.index:
                y_salida = float(normalizado.loc[op['Salida']])
                ax.scatter(
                    op['Salida'], y_salida, color=COLOR_SALIDA,
                    marker='X', s=170, edgecolor='black', linewidth=1.1,
                    zorder=7
                )

    # Formato común de la figura.
    ax.axhline(100, color='gray', ls='--', lw=1.5, alpha=.7)
    ax.set_title(
        f'V2.2 SOLO LONG — algoritmo original RSI + Bollinger — {meses} meses',
        fontsize=18, fontweight='bold', pad=14
    )
    ax.set_xlabel('Mes', fontsize=14, fontweight='bold')
    ax.set_ylabel('Precio normalizado (inicio = 100)', fontsize=14, fontweight='bold')
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.grid(alpha=.28, linewidth=.8)
    ax.tick_params(axis='both', labelsize=12)

    # Leyenda: líneas automáticas de activos y dos marcadores de operaciones.
    handles, labels = ax.get_legend_handles_labels()
    handles += [
        Line2D([0], [0], marker='^', color='w', markerfacecolor=COLOR_LONG,
               markeredgecolor='black', markersize=13, label='Entrada LONG'),
        Line2D([0], [0], marker='X', color='w', markerfacecolor=COLOR_SALIDA,
               markeredgecolor='black', markersize=12, label='Salida LONG')
    ]
    ax.legend(
        handles=handles, loc='best', ncol=min(3, len(handles)),
        fontsize=12, framealpha=.96, borderpad=1.0,
        handlelength=2.5, labelspacing=.8
    )
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    nombre = f'Backtest_V2_2_SOLO_LONG_precios_operaciones_{meses}m.png'
    fig.savefig(nombre, dpi=170, bbox_inches='tight')
    display(fig)
    plt.close(fig)

In [ ]:
# ============ GRÁFICO RESUMEN DE RENTABILIDAD SOLO LONG ============
# Todos los activos y la cartera total parten de 0 %, lo que permite comparar
# rentabilidades aunque los capitales iniciales asignados sean diferentes.
n = len(VENTANAS_MESES)
fig, axes = plt.subplots(n, 1, figsize=(19, max(7, 5.5 * n)), squeeze=False)

# BUCLE DE SUBGRÁFICAS: empareja cada eje con una ventana temporal activa.
for ax, meses in zip(axes[:, 0], VENTANAS_MESES):

    # BUCLE DE ACTIVOS: transforma cada curva monetaria en rentabilidad porcentual.
    for ticker in tickers:
        eq = resultados[meses][ticker]['Equity']['Equity']
        rentabilidad = (eq / capitales[ticker] - 1) * 100
        ax.plot(
            rentabilidad.index, rentabilidad, color=colores[ticker],
            lw=2.5, alpha=.92, label=ticker
        )

    # La línea negra representa el rendimiento agregado de toda la cartera.
    capital_total_inicial = sum(capitales.values())
    total = resultados[meses]['TOTAL CARTERA']['Equity']['Equity']
    rentabilidad_total = (total / capital_total_inicial - 1) * 100
    ax.plot(
        rentabilidad_total.index, rentabilidad_total,
        color='black', lw=4.5, label='TOTAL CARTERA', zorder=6
    )

    # Formato individual de cada subgráfica.
    ax.axhline(0, color='gray', ls='--', lw=1.4, alpha=.75)
    ax.set_title(
        f'Rentabilidad acumulada — ventana {meses} meses',
        fontsize=17, fontweight='bold', pad=12
    )
    ax.set_ylabel('Ganancia acumulada (%)', fontsize=13, fontweight='bold')
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.0f}%'))
    ax.grid(alpha=.28, linewidth=.8)
    ax.legend(
        loc='best', ncol=min(4, len(tickers) + 1),
        fontsize=12, framealpha=.96, borderpad=.9
    )
    ax.tick_params(axis='both', labelsize=11)
    ax.tick_params(axis='x', rotation=45)

# Etiqueta inferior compartida y exportación del resumen completo.
axes[-1, 0].set_xlabel('Mes', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(
    'Backtest_V2_2_SOLO_LONG_resumen_rentabilidad_porcentual.png',
    dpi=170, bbox_inches='tight'
)
display(fig)
plt.close(fig)

## Interpretación y límites

- Esta versión reproduce las reglas del algoritmo original señalado en `Cuaderno_Pruebas_META_y_otros.ipynb`, pero permite utilizarlas con cualquier activo configurado.
- La confirmación de rebote compara el cierre de la sesión actual con el cierre anterior y registra la compra en la apertura de esa misma sesión. Esta característica se conserva deliberadamente para mantener el comportamiento del algoritmo original.
- La estrategia nunca abre posiciones SHORT. Durante una bajada permanecerá en efectivo salvo que aparezca una señal LONG completa.
- `FECHA_FIN_VENTANA` determina el extremo final; si no hay cotización ese día, se utiliza la última sesión común anterior.
- Cada activo utiliza exclusivamente el capital indicado en `ACTIVOS`, sin transferencias ni rebalanceos.
- El efectivo no recibe intereses y el modelo no incluye comisiones, impuestos ni deslizamiento.
- Los rendimientos de activos internacionales permanecen en sus monedas locales; el total no corrige las variaciones de divisa.
- Los resultados históricos no garantizan rendimientos futuros.